In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import joblib

In [2]:
X_train = pd.read_csv("processed/X_train_processed.csv")
X_test = pd.read_csv("processed/X_test_processed.csv")

y_train = pd.read_csv("processed/y_train.csv").squeeze()
y_test = pd.read_csv("processed/y_test.csv").squeeze()

print(X_train.shape)
print(X_test.shape)

(5634, 65)
(1409, 65)


In [3]:
logistic = LogisticRegression(
    max_iter=2000,
    random_state=42
)

param_grid = {
    "C": np.logspace(-3, 2, 10),
    "class_weight": [
        None,
        "balanced",
        {0: 1, 1: 1.5},
        {0: 1, 1: 2},
        {0: 1, 1: 2.5}
    ],
    "solver": ["liblinear", "lbfgs"]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=logistic,
    param_distributions=param_grid,
    n_iter=15,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=LogisticRegression(max_iter=2000, random_state=42),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'C': array([1.00000000e-03, 3.59381366e-03, 1.29154967e-02, 4.64158883e-02,
       1.66810054e-01, 5.99484250e-01, 2.15443469e+00, 7.74263683e+00,
       2.78255940e+01, 1.00000000e+02]),
                                        'class_weight': [None, 'balanced',
                                                         {0: 1, 1: 1.5},
                                                         {0: 1, 1: 2},
                                                         {0: 1, 1: 2.5}],
                                        'solver': ['liblinear', 'lbfgs']},
                   random_state=42, scoring='f1')

In [4]:
print("Best parameters:")
print(random_search.best_params_)

print("\nBest CV F1:")
print(random_search.best_score_)

Best parameters:
{'solver': 'liblinear', 'class_weight': {0: 1, 1: 2.5}, 'C': 0.003593813663804626}

Best CV F1:
0.6346555056913505


In [5]:
tuned_model = random_search.best_estimator_

y_prob = tuned_model.predict_proba(X_test)[:, 1]
y_pred = tuned_model.predict(X_test)

print(classification_report(
    y_test,
    y_pred,
    target_names=["No Churn", "Churn"]
))

              precision    recall  f1-score   support

    No Churn       0.90      0.75      0.82      1035
       Churn       0.53      0.77      0.63       374

    accuracy                           0.76      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.76      0.77      1409



In [6]:
improved_results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "PR-AUC": average_precision_score(y_test, y_prob)
}

pd.DataFrame([improved_results])

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,0.757275,0.52952,0.76738,0.626638,0.845088,0.647608


In [7]:
threshold_results = []

for threshold in np.arange(0.30, 0.71, 0.05):

    predictions = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_test, predictions, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,Threshold,Accuracy,Precision,Recall,F1
0,0.30,0.645848,0.422935,0.917112,0.578903
1,0.35,0.671398,0.441370,0.895722,0.591350
2,0.40,0.712562,0.476762,0.850267,0.610951
3,0.45,0.741661,0.508333,0.815508,0.626283
4,0.50,0.757275,0.529520,0.767380,0.626638
5,0.55,0.773598,0.556008,0.729947,0.631214
6,0.60,0.789212,0.590588,0.671123,0.628285
7,0.65,0.799148,0.630372,0.588235,0.608575
8,0.70,0.806246,0.675958,0.518717,0.586989


In [8]:
joblib.dump(
    tuned_model,
    "../models/improved_logistic_regression.pkl"
)

print("Improved model saved.")

Improved model saved.


In [3]:
dt_model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

param_dist = {
    "max_depth": [3, 4, 5, 6, 7, 8, 10, None],
    "min_samples_split": [2, 5, 10, 15, 20],
    "min_samples_leaf": [1, 2, 5, 10, 15],
    "criterion": ["gini", "entropy"],
    "max_features": [None, "sqrt", "log2"]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_search_dt = RandomizedSearchCV(
    estimator=dt_model,
    param_distributions=param_dist,
    n_iter=30,
    scoring="f1",
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search_dt.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=DecisionTreeClassifier(class_weight='balanced',
                                                    random_state=42),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [3, 4, 5, 6, 7, 8, 10,
                                                      None],
                                        'max_features': [None, 'sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 5, 10, 15],
                                        'min_samples_split': [2, 5, 10, 15,
                                                              20]},
                   random_state=42, scoring='f1', verbose=1)

In [4]:
print("Best Parameters:")
print(random_search_dt.best_params_)

print("\nBest Cross-Validation F1:")
print(random_search_dt.best_score_)

Best Parameters:
{'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': None, 'max_depth': 5, 'criterion': 'entropy'}

Best Cross-Validation F1:
0.6210219913085064


In [5]:
improved_dt = random_search_dt.best_estimator_

print(improved_dt)

DecisionTreeClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=5, min_samples_leaf=10, random_state=42)


In [6]:
y_prob_dt = improved_dt.predict_proba(X_test)[:, 1]

y_pred_dt = (y_prob_dt >= 0.50).astype(int)

dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt)
dt_recall = recall_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)
dt_roc_auc = roc_auc_score(y_test, y_prob_dt)
dt_pr_auc = average_precision_score(y_test, y_prob_dt)

print("Improved Decision Tree Results")
print("--------------------------------")
print("Accuracy :", round(dt_accuracy, 4))
print("Precision:", round(dt_precision, 4))
print("Recall   :", round(dt_recall, 4))
print("F1 Score :", round(dt_f1, 4))
print("ROC-AUC  :", round(dt_roc_auc, 4))
print("PR-AUC   :", round(dt_pr_auc, 4))

Improved Decision Tree Results
--------------------------------
Accuracy : 0.7452
Precision: 0.5134
Recall   : 0.7701
F1 Score : 0.616
ROC-AUC  : 0.8349
PR-AUC   : 0.6235


In [7]:
threshold_results = []

thresholds = np.arange(0.30, 0.71, 0.05)

for threshold in thresholds:
    y_pred = (y_prob_dt >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,Threshold,Accuracy,Precision,Recall,F1
0,0.30,0.640170,0.419589,0.927807,0.577852
1,0.35,0.662881,0.435006,0.903743,0.587315
2,0.40,0.705465,0.468986,0.828877,0.599034
3,0.45,0.724627,0.488854,0.820856,0.612774
4,0.50,0.745209,0.513369,0.770053,0.616043
5,0.55,0.764372,0.541667,0.729947,0.621868
6,0.60,0.764372,0.541667,0.729947,0.621868
7,0.65,0.787083,0.593434,0.628342,0.610390
8,0.70,0.797729,0.628242,0.582888,0.604716


In [8]:
best_threshold_row = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

best_dt_threshold = best_threshold_row["Threshold"]

print("Selected Decision Tree Threshold:", best_dt_threshold)
print("\nBest Threshold Results:")
print(best_threshold_row)

Selected Decision Tree Threshold: 0.55

Best Threshold Results:
Threshold    0.550000
Accuracy     0.764372
Precision    0.541667
Recall       0.729947
F1           0.621868
Name: 5, dtype: float64


In [9]:
joblib.dump(
    improved_dt,
    "../models/improved_decision_tree.pkl"
)

print("Improved Decision Tree saved successfully!")

Improved Decision Tree saved successfully!


In [10]:
joblib.dump(
    best_dt_threshold,
    "../models/decision_tree_threshold.pkl"
)

print("Decision Tree threshold saved successfully!")

Decision Tree threshold saved successfully!
